# Trajectory 10 recurrence with BM4 and tighter Newton tolerances

This notebook isolates **trajectory 10** from the reproducible 40-trajectory campaign in `study_40_parallel_bm4_recurrences.ipynb`. It reconstructs the same 40-point initial-condition design, selects the point with zero-based index 9, and integrates that single particle with the current `BM4Implicit` API.

BM4 integrates the particle through 95 normalized cycles with 200 complete steps per cycle. This repetition reduces both Newton tolerances by two orders of magnitude relative to the preceding trajectory-10 experiment. At every cycle boundary $t=n$, the notebook measures the minimum-image periodic distance from the particle to its initial position. The coarse campaign identified cycle 8 as the nearest return for this trajectory, so that cycle is highlighted while all nonzero cycle boundaries are ranked independently at the requested resolution.

## Numerical protocol

The run uses **95 cycles and 200 fourth-order BM4 steps per cycle**, giving $\Delta t=0.005$ and 19,000 complete steps. Every integration step is saved, producing 19,001 aligned states; recurrence is evaluated only at exact cycle boundaries. Saving the full grid is required by the current nonlinear-work visualization API. The Newton tolerances are $10^{-16}$ absolute and $10^{-15}$ relative, exactly $10^2$ times smaller than the preceding $10^{-14}$/$10^{-13}$ experiment.

A return is classified as close when the periodic distance is at most one percent of the cell width. This threshold is explicit and can be changed with `recurrence_tolerance_fraction`. The tightened Newton tolerances remain deliberately aggressive for binary64 arithmetic, so the nonlinear-work diagnostics should be checked for roundoff-limited corrections. The notebook studies numerical recurrence and does not claim an exact mathematical periodic orbit.

In [ ]:
from pathlib import Path
from time import perf_counter
from types import SimpleNamespace

from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import numpy as np

from diagnostics.paths import find_project_root
from dynamics import GuidingCenterDynamics
from initial_conditions import GCInitialConfiguration
from potential import load_gc2d_h5_potential
from simulation import BM4Implicit, InitialValueProblem, SimulationRequest, simulate
from studies import latin_hypercube_gc_configuration_with_near_center
from visualization import (
    animate_implicit_method_trajectories,
    display_animation,
    display_records_table,
    plot_implicit_method_iterations,
)

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True})

## Reproducible physical, initial-condition, and numerical configuration

In [ ]:
# Measured, nondimensionalized GC2D potential used in the source campaign.
project_root = find_project_root(Path.cwd())
data_path = project_root / "data/potential/V1/PHI_2.h5"
magnetic_field = 1.5
characteristic_length = 0.06
mode_selection = (0, 1)
interpolation_order = 3

# Reconstruct the exact 40-point campaign and select labelled trajectory 10.
source_particle_count = 40
selected_trajectory_number = 10
selected_particle_index = selected_trajectory_number - 1
initial_condition_seed = 20260905
domain_margin_fraction = 0.05
near_center_offset_fraction = (0.08, -0.06)

# Requested integration and saved-state grid.
rho = 0.3
coupling_frequency = float(np.pi / 8.0)
cycle_duration = 1.0
final_cycle = 95
fine_steps_per_cycle = 200
saved_samples_per_cycle = fine_steps_per_cycle
integration_step = cycle_duration / fine_steps_per_cycle
save_interval = cycle_duration / saved_samples_per_cycle
t_span = (0.0, final_cycle * cycle_duration)
step_count = final_cycle * fine_steps_per_cycle
saved_sample_count = final_cycle * saved_samples_per_cycle + 1

# Newton thresholds are two orders tighter than in the preceding run.
baseline_newton_absolute_tolerance = 1e-14
baseline_newton_relative_tolerance = 1e-13
newton_tolerance_reduction_factor = 100.0
newton_absolute_tolerance = (
    baseline_newton_absolute_tolerance / newton_tolerance_reduction_factor
)
newton_relative_tolerance = (
    baseline_newton_relative_tolerance / newton_tolerance_reduction_factor
)
newton_max_iterations = 60
jacobian_relative_step = float(np.cbrt(np.finfo(float).eps))

# Recurrence definition and the candidate from the lower-resolution campaign.
recurrence_tolerance_fraction = 0.01
candidate_cycle = 8
nearest_cycles_to_report = 10
local_window_cycles = 4.0

if not data_path.is_file():
    raise FileNotFoundError(f"Measured HDF5 potential not found: {data_path}")
potential = load_gc2d_h5_potential(
    data_path,
    B=magnetic_field,
    characteristic_length=characteristic_length,
    indx=mode_selection,
    interpolation_order=interpolation_order,
)

source_configuration = latin_hypercube_gc_configuration_with_near_center(
    potential,
    particle_count=source_particle_count,
    seed=initial_condition_seed,
    center_offset_fraction=near_center_offset_fraction,
    domain_margin_fraction=domain_margin_fraction,
)
source_state = source_configuration.initial_state
assert source_state is not None
source_x, source_y = source_configuration.positions(source_state)
assert source_x.shape == source_y.shape == (source_particle_count,)
selected_x = float(source_x[selected_particle_index])
selected_y = float(source_y[selected_particle_index])
initial_configuration = GCInitialConfiguration.from_components(
    x=np.asarray([selected_x], dtype=float),
    y=np.asarray([selected_y], dtype=float),
)

dynamics = GuidingCenterDynamics(potential, rho=rho)
problem = InitialValueProblem(dynamics, initial_configuration)
request = SimulationRequest.uniform(
    t_span=t_span,
    max_step=integration_step,
    sample_count=saved_sample_count,
)

cycle_numbers = np.arange(final_cycle + 1, dtype=int)
cycle_times = cycle_numbers.astype(float) * cycle_duration
cycle_sample_indices = np.rint(cycle_times / save_interval).astype(int)
np.testing.assert_allclose(request.output_times[cycle_sample_indices], cycle_times)
assert selected_particle_index == 9
assert step_count == 19_000
assert saved_sample_count == 19_001
assert integration_step == 0.005
assert newton_absolute_tolerance == 1e-16
assert newton_relative_tolerance == 1e-15
assert problem.particle_count == 1
assert 0 < candidate_cycle <= final_cycle

cell_period = float(dynamics.effective_potential.grid.period)
recurrence_tolerance = recurrence_tolerance_fraction * cell_period
display(Markdown(
    f"**Resolved run:** trajectory `{selected_trajectory_number}` at "
    f"$({selected_x:.10f}, {selected_y:.10f})$, `{step_count}` steps of "
    f"`{integration_step:g}`, `{saved_sample_count}` saved states, cycle "
    f"boundaries `0..{final_cycle}`, and recurrence tolerance "
    f"`{recurrence_tolerance:.6g}` "
    f"(`{100 * recurrence_tolerance_fraction:g}%` of the cell width)."
))

## Isolated trajectory 10 initial point

In [ ]:
initial_state = initial_configuration.initial_state
assert initial_state is not None
np.testing.assert_allclose(initial_state, (selected_x, selected_y))

grid = dynamics.effective_potential.grid
field_at_zero = np.asarray(dynamics.effective_potential.evaluate_grid(0.0), dtype=float)
figure, axis = plt.subplots(figsize=(7.5, 6.5), constrained_layout=True)
image = axis.imshow(
    field_at_zero.T,
    origin="lower",
    extent=(grid.xmin, grid.xmin + grid.period, grid.ymin, grid.ymin + grid.period),
    cmap="RdBu_r",
    aspect="equal",
)
axis.scatter(
    selected_x, selected_y, s=190, marker="*", color="gold",
    edgecolor="black", label="Trajectory 10 initial point",
)
axis.set(title="Trajectory 10 selected for recurrence analysis", xlabel="$x$", ylabel="$y$")
axis.legend()
figure.colorbar(image, ax=axis, label="Effective potential at $t=0$")
plt.show()
display(Markdown(f"Initial position: **$({selected_x:.10f}, {selected_y:.10f})$**."))

## High-precision BM4 integration

The integration remains in its own cell so the recurrence analysis and presentation cells can be re-executed without repeating the 19,000-step BM4 run.

In [ ]:
method_builders = {
    "BM4Implicit": lambda: BM4Implicit(
        coupling_frequency=coupling_frequency,
        newton_absolute_tolerance=newton_absolute_tolerance,
        newton_relative_tolerance=newton_relative_tolerance,
        newton_max_iterations=newton_max_iterations,
        newton_jacobian_method="analytic",
        newton_jacobian_relative_step=jacobian_relative_step,
        nonlinear_solver="newton",
        progress=True,
    ),
}
solutions = {}
runtimes = {}

In [ ]:
method_name = "BM4Implicit"
print(f"Starting {method_name}: trajectory {selected_trajectory_number}, {step_count} steps.")
started = perf_counter()
solutions[method_name] = simulate(problem, method_builders[method_name](), request)
runtimes[method_name] = perf_counter() - started
print(f"Completed {method_name} in {runtimes[method_name]:.3f} s.")

In [ ]:
assert tuple(solutions) == ("BM4Implicit",)
bm4_diagnostics = solutions["BM4Implicit"].diagnostics
assert bm4_diagnostics["projection_solver_formulation"] == "bm4_implicit_reduced"
assert int(bm4_diagnostics["step_count"]) == step_count
assert np.asarray(bm4_diagnostics["nonlinear_iterations"]).shape == (step_count,)
assert all(np.array_equal(solution.t, request.output_times) for solution in solutions.values())

run_rows = tuple(
    SimpleNamespace(
        method=method_name,
        steps=int(solution.diagnostics["step_count"]),
        saved_states=solution.t.size,
        runtime=runtimes[method_name],
        mean_corrections=float(np.mean(solution.diagnostics["nonlinear_iterations"])),
        maximum_corrections=int(np.max(solution.diagnostics["nonlinear_iterations"])),
        maximum_residual_ratio=float(np.max(
            np.asarray(solution.diagnostics["nonlinear_residual_norms"])
            / np.asarray(solution.diagnostics["nonlinear_tolerances"])
        )),
    )
    for method_name, solution in solutions.items()
)
display_records_table(
    run_rows,
    columns=(
        ("method", "Method", None),
        ("steps", "Complete steps", "d"),
        ("saved_states", "Saved states", "d"),
        ("runtime", "Runtime [s]", ".3f"),
        ("mean_corrections", "Mean corrections/step", ".3f"),
        ("maximum_corrections", "Max corrections/step", "d"),
        ("maximum_residual_ratio", "Max residual/tolerance", ".3e"),
    ),
)

## Distance from the initial state at every cycle boundary

For a periodic cell of width $L$, each coordinate displacement is reduced to $[-L/2,L/2)$ before taking the Euclidean norm. This prevents a boundary crossing from being mistaken for a large physical displacement.

In [ ]:
def periodic_displacement_to_initial(solution, sample_indices, period):
    """Return minimum-image dx, dy, and distance from the initial point."""
    x, y = solution.positions()
    if x.shape[0] != 1 or y.shape != x.shape:
        raise ValueError("Recurrence analysis requires one planar particle.")
    selected_x_history = x[0, sample_indices]
    selected_y_history = y[0, sample_indices]
    dx = (selected_x_history - x[0, 0] + 0.5 * period) % period - 0.5 * period
    dy = (selected_y_history - y[0, 0] + 0.5 * period) % period - 0.5 * period
    return dx, dy, np.hypot(dx, dy)

cycle_displacements = {}
cycle_distances = {}
for method_name, solution in solutions.items():
    dx, dy, distance = periodic_displacement_to_initial(
        solution, cycle_sample_indices, cell_period
    )
    np.testing.assert_allclose(distance[0], 0.0, atol=1e-14)
    cycle_displacements[method_name] = (dx, dy)
    cycle_distances[method_name] = distance

# Cycle zero is the initial state itself and must not set the logarithmic scale.
plotted_cycles = cycle_numbers[1:]
distance_display_floor = np.finfo(float).eps * cell_period
figure, axis = plt.subplots(figsize=(12, 5.5), constrained_layout=True)
method_styles = {"BM4Implicit": dict(color="tab:green", marker="o")}
for method_name, distances in cycle_distances.items():
    axis.semilogy(
        plotted_cycles,
        np.maximum(distances[1:], distance_display_floor),
        label=method_name,
        markersize=4.0,
        linewidth=1.4,
        markevery=1,
        **method_styles[method_name],
    )
axis.axhline(recurrence_tolerance, color="black", linestyle="--", label="Close-return threshold")
axis.axvline(candidate_cycle, color="tab:blue", linestyle=":", linewidth=2.0, label=f"Candidate cycle {candidate_cycle}")
axis.set(
    title="Trajectory 10 periodic distance from its initial position",
    xlabel="Cycle boundary $n$ (time $t=n$)",
    ylabel="Minimum-image distance to the initial position",
)
axis.set_xticks(cycle_numbers[1:])
axis.legend(ncol=2)
plt.show()

## Ranked return candidates

Cycle zero is excluded. The table reports the ten closest later cycle boundaries and states whether each one satisfies the chosen return threshold.

In [ ]:
candidate_rows = []
for method_name, distances in cycle_distances.items():
    report_count = min(nearest_cycles_to_report, final_cycle)
    nearest_indices = np.argsort(distances[1:])[:report_count] + 1
    dx, dy = cycle_displacements[method_name]
    for rank, index in enumerate(nearest_indices, start=1):
        candidate_rows.append(SimpleNamespace(
            method=method_name,
            rank=rank,
            cycle=int(cycle_numbers[index]),
            dx=float(dx[index]),
            dy=float(dy[index]),
            distance=float(distances[index]),
            cell_fraction=float(distances[index] / cell_period),
            close_return="yes" if distances[index] <= recurrence_tolerance else "no",
        ))
display_records_table(
    tuple(candidate_rows),
    columns=(
        ("method", "Method", None),
        ("rank", "Rank", "d"),
        ("cycle", "Cycle", "d"),
        ("dx", "Periodic dx", ".5e"),
        ("dy", "Periodic dy", ".5e"),
        ("distance", "Distance", ".5e"),
        ("cell_fraction", "Cell-width fraction", ".5e"),
        ("close_return", "Within threshold", None),
    ),
)

## Detailed view around cycle 8

The next figure uses all saved samples in a window around the candidate return from the coarse campaign, rather than only the integer cycle boundaries. It shows whether the closest sampled approach occurs exactly at $t=8$ or slightly before or after it.

In [ ]:
all_sample_indices = np.arange(request.output_times.size, dtype=int)
full_distance_histories = {
    method_name: periodic_displacement_to_initial(
        solution, all_sample_indices, cell_period
    )[2]
    for method_name, solution in solutions.items()
}
window_mask = np.abs(request.output_times - candidate_cycle) <= local_window_cycles
figure, axis = plt.subplots(figsize=(11, 5), constrained_layout=True)
for method_name, distances in full_distance_histories.items():
    axis.plot(
        request.output_times[window_mask],
        distances[window_mask],
        label=method_name,
        color=method_styles[method_name]["color"],
        linewidth=1.8,
    )
window_start = max(0, int(candidate_cycle - local_window_cycles))
window_stop = min(final_cycle, int(candidate_cycle + local_window_cycles))
for cycle in range(window_start, window_stop + 1):
    axis.axvline(cycle, color="0.75", linewidth=0.7, zorder=0)
axis.axhline(recurrence_tolerance, color="black", linestyle="--", label="Close-return threshold")
axis.axvline(candidate_cycle, color="tab:blue", linestyle=":", linewidth=2.0)
axis.set(
    title=f"Trajectory 10 distance around candidate cycle {candidate_cycle}",
    xlabel="Time",
    ylabel="Minimum-image distance to the initial position",
)
axis.legend()
plt.show()

## Interactive trajectory animation

The numerical solution retains all 19,001 saved states. The player uses 201 uniformly spaced frames and provides **Play/Pause**, step, speed, loop-mode, and slider controls.

In [ ]:
animation_frames = 201
animation_fps = 10
# The animation-only DPI keeps the complete JavaScript player compact.
with plt.rc_context({"figure.dpi": 72}):
    recurrence_animation = animate_implicit_method_trajectories(
        dynamics.effective_potential,
        solutions,
        frames=animation_frames,
        interval=int(round(1000.0 / animation_fps)),
        repeat=True,
        title_family="trajectory 10 BM4 recurrence",
    )
    display_animation(
        recurrence_animation,
        embed_limit_mb=100.0,
        interactive=True,
    )

## Nonlinear-solver audit

This diagnostic confirms that trajectory 10 was accepted at the requested Newton tolerances throughout all 19,000 complete steps.

In [ ]:
plot_implicit_method_iterations(solutions)
plt.show()

## Data-driven recurrence conclusion

In [ ]:
conclusion_lines = []
for method_name, distances in cycle_distances.items():
    best_index = int(np.argmin(distances[1:]) + 1)
    close_indices = np.flatnonzero(distances[1:] <= recurrence_tolerance) + 1
    close_cycles = ", ".join(str(int(cycle_numbers[index])) for index in close_indices) or "none"
    conclusion_lines.extend((
        f"- **{method_name} at cycle {candidate_cycle}:** distance `{distances[candidate_cycle]:.6e}` "
        f"(`{distances[candidate_cycle] / cell_period:.6e}` cell widths).",
        f"- **{method_name} closest later cycle:** `{best_index}` with distance `{distances[best_index]:.6e}`.",
        f"- **{method_name} cycles inside the chosen threshold:** {close_cycles}.",
    ))
candidate_supported = cycle_distances["BM4Implicit"][candidate_cycle] <= recurrence_tolerance
conclusion_lines.append(
    f"- **Threshold verdict for cycle {candidate_cycle}:** "
    + (
        "BM4 classifies it as a close return."
        if candidate_supported
        else "BM4 does not classify it as a close return."
    )
)
display(Markdown("\n".join(conclusion_lines)))